# oracle_hr_demo — Pipeline Walkthrough

Demonstrates ingesting from an Oracle-style HR database with a **bronze-layer SQL filter**,
a silver join across three tables, and three gold aggregations.

| Layer | What happens |
|-------|--------------|
| Bronze | dlt reads `employees`, `departments`, `jobs` from SQLite (or Oracle) |
| | `filter:` pushes `WHERE department_id IN (10,20,60,80,90)` to the source |
| | `select:` drops `email` (PII) before data enters the lake |
| Silver | Type-cast each table; UDF joins all three into `employees_enriched` |
| Gold | Three aggregations: headcount by dept · salary by job · salary utilisation |

**Data layout after a full run:**
```
oracle_hr/
├── data/
│   ├── source/
│   │   └── oracle_hr.db          ← SQLite source (setup_db.py)
│   ├── bronze/
│   │   ├── employees/            ← dlt raw shards
│   │   ├── departments/          ← dlt raw shards
│   │   ├── jobs/                 ← dlt raw shards
│   │   ├── employees.parquet     ← 12 rows (3 in Purchasing/Shipping excluded)
│   │   ├── departments.parquet   ← 7 rows
│   │   └── jobs.parquet          ← 8 rows
│   ├── silver/
│   │   ├── employees.parquet
│   │   ├── departments.parquet
│   │   ├── jobs.parquet
│   │   └── employees_enriched.parquet   ← 12 rows × joined columns
│   └── gold/oracle_hr/
│       ├── headcount_by_department.parquet
│       ├── salary_by_job.parquet
│       └── salary_utilization.parquet
```

Run cells top to bottom.

## Setup — navigate to example root

In [1]:
import os
import polars as pl
from pathlib import Path

# ipynb/ → oracle_hr/ → oracle_hr_demo/
example_root = Path(os.getcwd()).parent.parent
os.chdir(example_root)
print(f'Working directory: {os.getcwd()}')

Working directory: /workspace/openmedallion/examples/oracle_hr_demo


## Seed — create the HR database

Creates `oracle_hr/data/source/oracle_hr.db` with:
- **15 employees** across 7 departments (12 pass the bronze filter — all in target depts, ACTIVE and INACTIVE; 3 in Purchasing/Shipping are excluded)
- **7 departments** (5 in scope: Admin, Marketing, IT, Sales, Executive)
- **8 jobs** with salary bands (used by the gold pre-agg UDF)

In [2]:
!python setup_db.py

✅  Database seeded at oracle_hr/data/oracle_hr.db

   15 employees inserted:
    • 12 ACTIVE in target departments  (10, 20, 60, 80, 90)
    •  2 INACTIVE in target departments → removed by status filter
    •  3 ACTIVE in dept 30 / 50        → removed by department filter

   After bronze filter, pipeline ingests 12 employees.

Next steps:
  medallion run oracle_hr --projects . --layer bronze
  medallion run oracle_hr --projects . --layer silver
  medallion run oracle_hr --projects .

Or open oracle_hr/ipynb/walkthrough.ipynb for a guided run.


## Inspect source data

In [ ]:
import sqlite3

con = sqlite3.connect('oracle_hr/data/source/oracle_hr.db')

print('── employees (15 total in DB) ──')
print(pl.read_database('SELECT * FROM employees ORDER BY department_id, employee_id', con))

print('\n── departments ──')
print(pl.read_database('SELECT * FROM departments ORDER BY department_id', con))

print('\n── jobs ──')
print(pl.read_database('SELECT * FROM jobs ORDER BY job_id', con))

con.close()

---
## Bronze — filtered ingestion

The `filter` field in `backend/bronze.yaml` applies:
```
WHERE department_id IN (10, 20, 60, 80, 90)
```
This is pushed directly to the SQL query — rows in dept 30 (Purchasing) and dept 50 (Shipping)
never enter the data lake. Both ACTIVE and INACTIVE employees in the target departments are ingested.

The `select` field drops `email` (PII) at source — it never appears in bronze or downstream.

In [4]:
!medallion run oracle_hr --layer bronze


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  medallion  ·  oracle_hr  ·  bronze ingestion
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋  [config] bronze: oracle_hr/backend/bronze.yaml
📋  [config] silver: oracle_hr/backend/silver.yaml
📋  [config] gold  : oracle_hr/backend/gold.yaml

── Bronze ─────────────────────────────────────────────────
2026-05-06 18:50:56,191|[WARNING]|44|126946220335232|dlt|filesystem.py|prepare_load_table:860|Falling back to `append` on `jobs`.
2026-05-06 18:50:56,191|[WARNING]|44|126946220335232|dlt|filesystem.py|prepare_load_table:860|Falling back to `append` on `departments`.
2026-05-06 18:50:56,192|[WARNING]|44|126946220335232|dlt|filesystem.py|prepare_load_table:860|Falling back to `append` on `departments`.
2026-05-06 18:50:56,192|[WARNING]|44|126946220335232|dlt|filesystem.py|prepare_load_table:860|Falling back to `append` on `jobs`.
2026-05-06 18:50:56,237|[WARNING]|44|126946220335232|dlt|filesystem.py|prepare_load_table:

In [ ]:
bronze_dir = Path('data/bronze')

print('── employees.parquet (bronze — after filter) ──')
emp_bronze = pl.read_parquet(bronze_dir / 'employees.parquet')
print(f'Shape: {emp_bronze.shape}  ← 12 rows (not 15; 3 in Purchasing/Shipping excluded)')
print(emp_bronze.select(['employee_id','first_name','last_name','department_id','hire_date','salary']))

print(f'\nUnique department_id values: {sorted(emp_bronze["department_id"].unique().to_list())}')

In [ ]:
print('── departments.parquet (all 7 rows — no filter on this table) ──')
print(pl.read_parquet(bronze_dir / 'departments.parquet')
        .select(['department_id','department_name']))

print('\n── jobs.parquet (all 8 rows) ──')
print(pl.read_parquet(bronze_dir / 'jobs.parquet')
        .select(['job_id','job_title','min_salary','max_salary']))

---
## Silver — type-cast + enrichment join

Phase 1: cast columns to correct types.  
Phase 2: UDF (`build_employees_enriched`) joins employees → departments → jobs into one wide table.

In [12]:
!medallion run oracle_hr --projects . --layer silver


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  medallion  ·  oracle_hr  ·  bronze → silver
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋  [config] bronze: oracle_hr/backend/bronze.yaml
📋  [config] silver: oracle_hr/backend/silver.yaml
📋  [config] gold  : oracle_hr/backend/gold.yaml

  ⏭️  bronze  skipped (existing files)

── Silver ─────────────────────────────────────────────────
🔧  [silver] base    employees.parquet → employees.parquet  (15 rows)
🔧  [silver] base    departments.parquet → departments.parquet  (7 rows)
🔧  [silver] base    jobs.parquet → jobs.parquet  (8 rows)
🔧  [silver] derived employees_enriched.parquet  (15 rows)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅  bronze → silver complete.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



In [ ]:
silver_dir = Path('data/silver')

print('── employees_enriched.parquet (derived table) ──')
enriched = pl.read_parquet(silver_dir / 'employees_enriched.parquet')
print(f'Shape: {enriched.shape}')
print(enriched.select([
    'first_name', 'last_name', 'salary',
    'department_name', 'job_title',
    'min_salary', 'max_salary'
]).sort('department_name', 'salary', descending=[False, True]))

---
## Gold — three aggregations

1. **headcount_by_department** — headcount, total payroll, avg salary per department  
2. **salary_by_job** — min/max/avg actual salary per job title  
3. **salary_utilization** — avg salary as % of job-band max, per dept × job (pre-agg UDF adds `salary_pct_of_max`)

In [14]:
!medallion run oracle_hr --projects .


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  medallion  ·  oracle_hr  ·  bronze → silver → gold
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋  [config] bronze: oracle_hr/backend/bronze.yaml
📋  [config] silver: oracle_hr/backend/silver.yaml
📋  [config] gold  : oracle_hr/backend/gold.yaml

  ⏭️  bronze  skipped (existing files)
  ⏭️  silver  skipped (existing files)

── Gold ───────────────────────────────────────────────────
📊  [gold/oracle_hr] headcount_by_department.parquet  (7 rows)
📊  [gold/oracle_hr] salary_by_job.parquet  (8 rows)
⚙️   [gold]  udf add_salary_metrics()  15 → 15 rows
📊  [gold/oracle_hr] salary_utilization.parquet  (10 rows)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅  bronze → silver → gold complete.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



In [ ]:
gold_dir = Path('data/gold/oracle_hr')

print('── headcount_by_department.parquet ──')
hc = pl.read_parquet(gold_dir / 'headcount_by_department.parquet')
print(hc.sort('total_payroll', descending=True))

print(f'\nTotal headcount across departments: {hc["headcount"].sum()}')
print(f'Total payroll: ${hc["total_payroll"].sum():,.0f}')

In [ ]:
print('── salary_by_job.parquet ──')
print(pl.read_parquet(gold_dir / 'salary_by_job.parquet')
        .sort('avg_actual_salary', descending=True))

In [ ]:
print('── salary_utilization.parquet ──')
print('(avg salary as % of job-band maximum, grouped by dept × job)')
util = pl.read_parquet(gold_dir / 'salary_utilization.parquet')
print(util.sort('avg_salary_pct_of_max', descending=True))

---
## Connecting to a real Oracle / Postgres database

1. Edit `examples/secrets.yaml` (one level above this demo) — fill in your `oracle:` or `postgres:` block.  
   Copy from `examples/secrets.yaml.example` if it doesn't exist yet.
2. In `oracle_hr/backend/bronze.yaml`, comment out `connection_string` and uncomment `credentials_file: ../secrets.yaml` + `dialect: oracle` (or `postgres`, `mysql`, `mssql`).
3. Install the driver if needed:

```bash
pip install "openmedallion[oracle]"   # for Oracle
```

4. Re-run bronze:

```bash
medallion run oracle_hr --projects . --layer bronze
```

The `filter:` in `bronze.yaml` passes straight through to the Oracle query — no code changes needed.

## Things to Try

- **Change the department filter**: edit `backend/bronze.yaml` → `filter:` field, re-run bronze
- **Add a new employee**: insert a row into SQLite, then `rm -rf oracle_hr/data/bronze/` and re-run bronze
- **Add a new gold aggregation**: add a YAML block to `backend/gold.yaml` (e.g., avg salary by department and job)
- **Use a real Oracle DB**: follow the "Connecting to a real Oracle / Postgres database" section above